In [41]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)

In [ ]:
# Read the CSV file into a pandas DataFrame.
# The CSV file must be in the same folder as this Jupyter notebook.
df = pd.read_csv("cleaned_Charlotte_traffic.csv")

# Convert timestamp strings into datetime objects.
# This is required so we can correctly sort the data by time later.
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Print a basic summary to confirm that the data loaded correctly.
print("Dataset loaded successfully.\n")
print(f"Number of rows: {len(df)}")
print(f"Number of columns: {len(df.columns)}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Number of locations: {df['location'].nunique()}")

# Display the first five rows of the dataset.
display(df.head())

Dataset loaded successfully.

Number of rows: 100
Number of columns: 29
Date range: 2026-09-26 12:17:53 to 2026-09-26 13:03:33
Number of locations: 25


,timestamp,date,day_of_week,hour,minute,is_weekend,is_rush_hour,location,road,road_type,...,delay_s,confidence,road_closure,speed_ratio,temperature_f,precipitation_in,is_raining,weather_code,wind_speed_mph,traffic_level
0,2026-09-26 12:17:53,2026-09-26,Saturday,12,17,True,False,I-77 at Arrowood Rd,I-77,interstate,...,68,1,False,0.34,70.8,0,False,0,6.5,high
1,2026-09-26 12:17:53,2026-09-26,Saturday,12,17,True,False,I-77 at Tyvola Rd,I-77,interstate,...,99,1,False,0.55,70.8,0,False,0,6.5,medium
2,2026-09-26 12:17:53,2026-09-26,Saturday,12,17,True,False,I-77 at John Belk Fwy,I-77,interstate,...,135,1,False,0.26,70.8,0,False,0,6.5,high
3,2026-09-26 12:17:53,2026-09-26,Saturday,12,17,True,False,I-77 at LaSalle St,I-77,interstate,...,0,1,False,1.00,70.8,0,False,0,6.5,low
4,2026-09-26 12:17:53,2026-09-26,Saturday,12,17,True,False,I-77 at Sunset Rd,I-77,interstate,...,0,1,False,1.00,70.8,0,False,0,6.5,low


In [43]:
# Count the number of records in each traffic category.
# This shows whether the data is balanced across low, medium, and high traffic.
print("Traffic level counts:")
print(df["traffic_level"].value_counts())

# Show each traffic class as a percentage of all records.
# This helps identify class imbalance, such as many more low-traffic rows.
print("\nTraffic level percentages:")
print((df["traffic_level"].value_counts(normalize=True) * 100).round(2))

Traffic level counts:
traffic_level
low       78
medium    14
high       8
Name: count, dtype: int64

Traffic level percentages:
traffic_level
low       78.0
medium    14.0
high       8.0
Name: proportion, dtype: float64


In [44]:
# Define the input columns that the model will use for prediction.
feature_cols = [
    "hour",
    "minute",
    "is_weekend",
    "is_rush_hour",
    "road",
    "road_type",
    "road_class_frc",
    "temperature_f",
    "precipitation_in",
    "is_raining",
    "weather_code",
    "wind_speed_mph",
    "road_closure",
    "location",
]

# Create a copy of the selected feature columns.
X = df[feature_cols].copy()

# Define columns containing True/False values.
boolean_cols = [
    "is_weekend",
    "is_rush_hour",
    "is_raining",
    "road_closure",
]

# Convert Boolean values to numeric values: True becomes 1 and False becomes 0.
for col in boolean_cols:
    X[col] = X[col].astype(int)

# Create a binary target:
# 0 = low traffic, 1 = medium or high traffic.
y = (df["traffic_level"] != "low").astype(int)

# Print the size of the feature matrix.
print("Feature matrix shape:", X.shape)

# Print the count of low-traffic and traffic rows.
print("\nTarget distribution:")
print(
    y.value_counts().rename(
        index={
            0: "No/low traffic",
            1: "Traffic",
        }
    )
)

Feature matrix shape: (100, 14)

Target distribution:
traffic_level
No/low traffic    78
Traffic           22
Name: count, dtype: int64


In [45]:
# Define text-based columns that must be one-hot encoded.
categorical_cols = [
    "road",
    "road_type",
    "road_class_frc",
    "location",
]

# Treat all remaining selected feature columns as numeric.
numeric_cols = [
    col for col in X.columns
    if col not in categorical_cols
]

# Print categorical feature names.
print("Categorical columns:")
print(categorical_cols)

# Print numeric feature names.
print("\nNumeric columns:")
print(numeric_cols)

Categorical columns:
['road', 'road_type', 'road_class_frc', 'location']

Numeric columns:
['hour', 'minute', 'is_weekend', 'is_rush_hour', 'temperature_f', 'precipitation_in', 'is_raining', 'weather_code', 'wind_speed_mph', 'road_closure']


In [46]:
# Create a transformer that preprocesses categorical and numeric columns separately.
preprocessor = ColumnTransformer(
    transformers=[
        (
            # One-hot encode text-based feature columns.
            # Ignore unknown categories if new data has unseen values.
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols,
        ),
        (
            # Standardize numeric feature columns.
            "numeric",
            StandardScaler(),
            numeric_cols,
        ),
    ]
)

# Confirm that preprocessing has been configured.
print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


In [47]:
# Sort all data by timestamp so earlier rows are used for training
# and later rows are used for testing.
df_sorted = df.sort_values("timestamp")

# Calculate the row where the first 80% of the dataset ends.
split_index = int(len(df_sorted) * 0.80)

# Save the row indices for the earlier training data.
train_indices = df_sorted.index[:split_index]

# Save the row indices for the later test data.
test_indices = df_sorted.index[split_index:]

# Select feature values for the training rows.
X_train = X.loc[train_indices]

# Select feature values for the testing rows.
X_test = X.loc[test_indices]

# Select target labels for the training rows.
y_train = y.loc[train_indices]

# Select target labels for the testing rows.
y_test = y.loc[test_indices]

# Print the number of rows in each dataset split.
print(f"Training rows: {len(X_train)}")
print(f"Testing rows: {len(X_test)}")

# Print the timestamp range used to train the model.
print("\nTraining time range:")
print(
    f"{df.loc[train_indices, 'timestamp'].min()} "
    f"to {df.loc[train_indices, 'timestamp'].max()}"
)

# Print the timestamp range used to test the model.
print("\nTesting time range:")
print(
    f"{df.loc[test_indices, 'timestamp'].min()} "
    f"to {df.loc[test_indices, 'timestamp'].max()}"
)

Training rows: 80
Testing rows: 20

Training time range:
2026-09-26 12:17:53 to 2026-09-26 13:03:33

Testing time range:
2026-09-26 13:03:33 to 2026-09-26 13:03:33


In [48]:
# Build a pipeline that preprocesses the data and then trains the classifier.
model = Pipeline(
    steps=[
        # Apply the categorical and numeric preprocessing steps.
        (
            "preprocessor",
            preprocessor,
        ),
        # Train a Random Forest model using the preprocessed features.
        (
            "random_forest",
            RandomForestClassifier(
                # Use 300 decision trees in the forest.
                n_estimators=300,

                # Limit each tree's depth.
                max_depth=8,

                # Require at least two samples in each final leaf.
                min_samples_leaf=2,

                # Balance class weights because low traffic is more common.
                class_weight="balanced",

                # Keep results reproducible across runs.
                random_state=42,

                # Use available processor cores.
                n_jobs=-1,
            ),
        ),
    ]
)

# Confirm that the model pipeline was created.
print("Random Forest pipeline created successfully.")

Random Forest pipeline created successfully.


In [49]:
# Fit preprocessing and train the Random Forest model using training data.
model.fit(X_train, y_train)

# Print a confirmation when training is finished.
print("Model training complete.")

Model training complete.


In [50]:
# Predict traffic classifications for the held-out test data.
y_pred = model.predict(X_test)

# Calculate the overall fraction of correct predictions.
accuracy = accuracy_score(y_test, y_pred)

# Calculate balanced accuracy, which gives equal importance to both classes.
balanced_accuracy = balanced_accuracy_score(y_test, y_pred)

# Print main evaluation metrics.
print("=== Model Evaluation ===")
print(f"Accuracy: {accuracy:.3f}")
print(f"Balanced accuracy: {balanced_accuracy:.3f}")

# Print detailed precision, recall, and F1-score results.
print("\n=== Classification Report ===")
print(
    classification_report(
        y_test,
        y_pred,
        labels=[0, 1],
        target_names=["No/low traffic", "Traffic"],
        zero_division=0,
    )
)

# Create a labeled table of actual versus predicted classes.
confusion_df = pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=[0, 1]),
    index=["Actual no/low traffic", "Actual traffic"],
    columns=["Predicted no/low traffic", "Predicted traffic"],
)

# Display the confusion matrix.
print("=== Confusion Matrix ===")
display(confusion_df)

=== Model Evaluation ===
Accuracy: 0.950
Balanced accuracy: 0.967

=== Classification Report ===
                precision    recall  f1-score   support

No/low traffic       1.00      0.93      0.97        15
       Traffic       0.83      1.00      0.91         5

      accuracy                           0.95        20
     macro avg       0.92      0.97      0.94        20
  weighted avg       0.96      0.95      0.95        20

=== Confusion Matrix ===


,Predicted no/low traffic,Predicted traffic
Actual no/low traffic,14,1
Actual traffic,0,5


In [51]:
# Select original information from the test rows for comparison.
test_results = df.loc[
    test_indices,
    [
        "timestamp",
        "location",
        "road",
        "traffic_level",
        "current_speed_mph",
        "free_flow_speed_mph",
        "delay_s",
    ],
].copy()

# Add readable actual labels based on the binary target.
test_results["actual_traffic"] = y_test.map(
    {
        0: "No/low traffic",
        1: "Traffic",
    }
)

# Add readable predicted labels from the model predictions.
test_results["predicted_traffic"] = pd.Series(
    y_pred,
    index=test_results.index,
).map(
    {
        0: "No/low traffic",
        1: "Traffic",
    }
)

# Add the model's probability that traffic is present.
test_results["traffic_probability"] = model.predict_proba(X_test)[:, 1]

# Sort the test results from highest to lowest traffic probability.
test_results = test_results.sort_values(
    by="traffic_probability",
    ascending=False,
)

# Display prediction results and format probabilities as percentages.
display(
    test_results.style.format(
        {
            "traffic_probability": "{:.1%}",
        }
    )
)

,timestamp,location,road,traffic_level,current_speed_mph,free_flow_speed_mph,delay_s,actual_traffic,predicted_traffic,traffic_probability
98,2026-09-26 13:03:33,W.T. Harris Blvd at Albemarle,W.T. Harris Blvd,medium,17,29,60,Traffic,Traffic,66.1%
76,2026-09-26 13:03:33,I-77 at Tyvola Rd,I-77,medium,37,64,88,Traffic,Traffic,65.3%
77,2026-09-26 13:03:33,I-77 at John Belk Fwy,I-77,medium,40,65,29,Traffic,Traffic,64.2%
75,2026-09-26 13:03:33,I-77 at Arrowood Rd,I-77,high,21,64,75,Traffic,Traffic,64.0%
94,2026-09-26 13:03:33,South Blvd at Woodlawn Rd,South Blvd,medium,17,25,29,Traffic,Traffic,63.8%
96,2026-09-26 13:03:33,N Tryon St at University,N Tryon St,low,29,29,0,No/low traffic,Traffic,51.4%
78,2026-09-26 13:03:33,I-77 at LaSalle St,I-77,low,63,63,0,No/low traffic,No/low traffic,43.0%
79,2026-09-26 13:03:33,I-77 at Sunset Rd,I-77,low,70,70,0,No/low traffic,No/low traffic,42.2%
95,2026-09-26 13:03:33,Providence Rd at Sharon Amity,Providence Rd,low,37,37,0,No/low traffic,No/low traffic,25.8%
99,2026-09-26 13:03:33,Wilkinson Blvd,Wilkinson Blvd,low,41,41,0,No/low traffic,No/low traffic,23.0%


In [52]:
# Sort by timestamp, group by location, and keep the newest row
# available for each location.
latest_by_location = (
    df.sort_values("timestamp")
    .groupby("location", as_index=False)
    .tail(1)
    .copy()
)

# Select the same feature columns used during training.
X_latest = latest_by_location[feature_cols].copy()

# Convert Boolean feature columns to numeric values.
for col in boolean_cols:
    X_latest[col] = X_latest[col].astype(int)

# Predict the probability that each latest location record has traffic.
latest_by_location["traffic_probability"] = model.predict_proba(X_latest)[:, 1]

# Convert predicted probabilities into readable labels using a 50% cutoff.
latest_by_location["prediction"] = latest_by_location[
    "traffic_probability"
].apply(
    lambda probability: (
        "Traffic likely"
        if probability >= 0.50
        else "Low traffic likely"
    )
)

# Select the output columns to include in the future prediction table.
future_predictions = latest_by_location[
    [
        "location",
        "road",
        "timestamp",
        "traffic_level",
        "current_speed_mph",
        "free_flow_speed_mph",
        "delay_s",
        "traffic_probability",
        "prediction",
    ]
].sort_values(
    by="traffic_probability",
    ascending=False,
)

# Display predictions sorted from highest to lowest traffic probability.
display(
    future_predictions.style.format(
        {
            "traffic_probability": "{:.1%}",
        }
    )
)

,location,road,timestamp,traffic_level,current_speed_mph,free_flow_speed_mph,delay_s,traffic_probability,prediction
98,W.T. Harris Blvd at Albemarle,W.T. Harris Blvd,2026-09-26 13:03:33,medium,17,29,60,66.1%,Traffic likely
76,I-77 at Tyvola Rd,I-77,2026-09-26 13:03:33,medium,37,64,88,65.3%,Traffic likely
77,I-77 at John Belk Fwy,I-77,2026-09-26 13:03:33,medium,40,65,29,64.2%,Traffic likely
75,I-77 at Arrowood Rd,I-77,2026-09-26 13:03:33,high,21,64,75,64.0%,Traffic likely
94,South Blvd at Woodlawn Rd,South Blvd,2026-09-26 13:03:33,medium,17,25,29,63.8%,Traffic likely
96,N Tryon St at University,N Tryon St,2026-09-26 13:03:33,low,29,29,0,51.4%,Traffic likely
78,I-77 at LaSalle St,I-77,2026-09-26 13:03:33,low,63,63,0,43.0%,Low traffic likely
79,I-77 at Sunset Rd,I-77,2026-09-26 13:03:33,low,70,70,0,42.2%,Low traffic likely
95,Providence Rd at Sharon Amity,Providence Rd,2026-09-26 13:03:33,low,37,37,0,25.8%,Low traffic likely
99,Wilkinson Blvd,Wilkinson Blvd,2026-09-26 13:03:33,low,41,41,0,23.0%,Low traffic likely


In [53]:
# Keep only locations with at least a 50% predicted traffic probability.
high_risk_locations = future_predictions[
    future_predictions["traffic_probability"] >= 0.50
].copy()

# Print the number of locations predicted to have traffic.
print("Locations predicted to have traffic:")
print(f"Number of locations: {len(high_risk_locations)}\n")

# Display only the high-risk traffic locations.
display(
    high_risk_locations.style.format(
        {
            "traffic_probability": "{:.1%}",
        }
    )
)

Locations predicted to have traffic:
Number of locations: 6



,location,road,timestamp,traffic_level,current_speed_mph,free_flow_speed_mph,delay_s,traffic_probability,prediction
98,W.T. Harris Blvd at Albemarle,W.T. Harris Blvd,2026-09-26 13:03:33,medium,17,29,60,66.1%,Traffic likely
76,I-77 at Tyvola Rd,I-77,2026-09-26 13:03:33,medium,37,64,88,65.3%,Traffic likely
77,I-77 at John Belk Fwy,I-77,2026-09-26 13:03:33,medium,40,65,29,64.2%,Traffic likely
75,I-77 at Arrowood Rd,I-77,2026-09-26 13:03:33,high,21,64,75,64.0%,Traffic likely
94,South Blvd at Woodlawn Rd,South Blvd,2026-09-26 13:03:33,medium,17,25,29,63.8%,Traffic likely
96,N Tryon St at University,N Tryon St,2026-09-26 13:03:33,low,29,29,0,51.4%,Traffic likely
